In [42]:
!pip install pyspark

In [43]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

# Q1

In [44]:
!wget -q https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/quiz/sherlock.txt

In [45]:
rdd = sc.textFile("sherlock.txt")

In [46]:
rdd.take(5)

['A STUDY IN SCARLET.', '', '', '', '']

In [47]:
words = rdd.flatMap(lambda txt: txt.split(' '))
words.take(5)

['A', 'STUDY', 'IN', 'SCARLET.', '']

In [48]:
filtered = words.filter(lambda w: len(w) >= 2)  # Example에서 IN도 포함하므로 2글자인 것도 포함
filtered.take(5)

['STUDY', 'IN', 'SCARLET.', 'PART', 'I.']

In [49]:
heads = filtered.map(lambda w: (w, w.lower()[:2]))
heads.take(5)

[('STUDY', 'st'),
 ('IN', 'in'),
 ('SCARLET.', 'sc'),
 ('PART', 'pa'),
 ('I.', 'i.')]

In [50]:
heads_count = filtered.map(lambda w: (w.lower()[:2], 1))
heads_count.take(5)

[('st', 1), ('in', 1), ('sc', 1), ('pa', 1), ('i.', 1)]

In [51]:
counts = heads_count.reduceByKey(lambda x, y: x + y)
counts.take(5)

[('i.', 9), ('(b', 1), ('of', 1256), ('h.', 2), ('m.', 2)]

In [52]:
under32 = counts.filter(lambda x: x[1] < 32)
under32.take(5)

[('i.', 9), ('(b', 1), ('h.', 2), ('m.', 2), ('18', 2)]

In [53]:
under32.map(lambda x: x[1]).reduce(lambda x, y: x + y)

830

In [54]:
# sol
rdd.flatMap(lambda txt: txt.split(' ')) \
    .filter(lambda w: len(w) >= 2) \
    .map(lambda w: (w.lower()[:2], 1)) \
    .reduceByKey(lambda x, y: x + y) \
    .filter(lambda x: x[1] < 32) \
    .map(lambda x: x[1]) \
    .reduce(lambda x, y: x + y)

830

# Q2

In [55]:
!wget -q -O int-pair-input.txt https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/quiz/int-pair-input.txt

rdd = sc.textFile("int-pair-input.txt")
rdd.take(5)

['(2, 96)', '(19, 78)', '(13, 83)', '(17, 78)', '(32, 86)']

In [56]:
pairs = rdd.map(lambda line: line.replace("(", "").replace(")", "").split(",")) \
           .map(lambda x: (int(x[0]), int(x[1])))

a = pairs.first()[0]

b = pairs.map(lambda x: x[0]) \
         .reduce(lambda x, y: x if x > y else y)

c = pairs.map(lambda x: x[0]) \
         .reduce(lambda x, y: x if x < y else y)

d = pairs.filter(lambda x: x[0] == 1).count()

e = pairs.filter(lambda x: x[0] == 2).count()

print(a, b, c, d, e)
print(a + b + c + d + e)

2 32 0 31 28
93


# Q3

In [57]:
!git clone https://github.com/nongaussian/class-2024-datamining.git

fatal: destination path 'class-2024-datamining' already exists and is not an empty directory.


In [58]:
path = "/content/class-2024-datamining/data/World Hapiness Report/"

rdd15 = sc.textFile(path + "2015.csv")
rdd16 = sc.textFile(path + "2016.csv")
rdd17 = sc.textFile(path + "2017.csv")
rdd18 = sc.textFile(path + "2018.csv")
rdd19 = sc.textFile(path + "2019.csv")

In [59]:
print(rdd15.first())
print(rdd16.first())
print(rdd17.first())
print(rdd18.first())
print(rdd19.first())

Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)
Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)
Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Generosity,Trust (Government Corruption)
Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)
Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)


In [60]:
r15 = rdd15.filter(lambda x: not x.startswith("Country")) \
           .map(lambda x: x.split(",")) \
           .map(lambda x: (x[0], float(x[2])))

r16 = rdd16.filter(lambda x: not x.startswith("Country")) \
           .map(lambda x: x.split(",")) \
           .map(lambda x: (x[0], float(x[2])))

r17 = rdd17.filter(lambda x: not x.startswith("Country")) \
           .map(lambda x: x.split(",")) \
           .map(lambda x: (x[0], float(x[2])))

r18 = rdd18.filter(lambda x: not x.startswith("Country")) \
           .map(lambda x: x.split(",")) \
           .map(lambda x: (x[0], float(x[2])))

r19 = rdd19.filter(lambda x: not x.startswith("Country")) \
           .map(lambda x: x.split(",")) \
           .map(lambda x: (x[0], float(x[2])))

In [61]:
result = r15.join(r16) \
    .filter(lambda x: x[1][0] <= x[1][1]) \
    .map(lambda x: (x[0], x[1][1])) \
    .join(r17) \
    .filter(lambda x: x[1][0] <= x[1][1]) \
    .map(lambda x: (x[0], x[1][1])) \
    .join(r18) \
    .filter(lambda x: x[1][0] <= x[1][1]) \
    .map(lambda x: (x[0], x[1][1])) \
    .join(r19) \
    .filter(lambda x: x[1][0] <= x[1][1]) \
    .map(lambda x: x[0])

result.collect()

['Gabon',
 'Malta',
 'Senegal',
 'Chad',
 'Czech Republic',
 'Cambodia',
 'Tajikistan',
 'Congo (Brazzaville)',
 'Latvia',
 'Finland',
 'Dominican Republic',
 'Honduras',
 'Syria',
 'Estonia',
 'Mongolia',
 'Poland',
 'Togo',
 'Slovakia',
 'Hungary',
 'Cameroon',
 'Romania',
 'Benin',
 'Burkina Faso',
 'Serbia',
 'Ivory Coast',
 'Philippines',
 'Portugal',
 'Niger']